# Préparation des données SFT — MediQAl (mcqm)

Complément français du SFT : questions de QCM à **réponses multiples** (configuration `mcqm` de MediQAl, licence CC-BY-4.0), dont la réponse attendue est la **liste des bonnes réponses**.

- **Pourquoi** : le SFT `mcqu` n'apprend que des réponses uniques. Le DPO français (`data_prep_DPO_MediQAl_mcqm`) compare des listes : le modèle doit avoir appris ce format avant le DPO, qui ajuste des préférences mais n'enseigne pas un nouveau format.
- **Pas de recouvrement avec le DPO** : le DPO utilise les questions `mcqm` rattachées à un cas clinique ; ce notebook utilise les questions **sans cas clinique**. Un contrôle final le vérifie.
- **Sortie** : `data/anonymized/sft_mediqal_mcqm_{train,validation,test}.anonymized.jsonl`, au format `messages` du SFT.

# Import

In [ ]:
import json
import re

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

from projet_14.rgpd import run

SEED = 42
LETTERS = list("abcde")

# Options trop courtes (« 1 », « Oui »...) : peu informatives sans la question complète
MIN_OPTION_CHARS = 3

# Taille maximale des jeux (complément du SFT mcqu : quelques centaines à un millier d'exemples)
N_TRAIN = 1000
N_VALIDATION = 200
N_TEST = 500

# Get hugging face dataset

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqm")
splits = {name: ds[name].to_pandas() for name in ds}

# Analyse dataset

In [ ]:
splits["train"].head()

In [ ]:
# Questions avec et sans cas clinique : le DPO utilise les premières, ce notebook les secondes
for name, df in splits.items():
    print(f"{name}: {len(df)} rows, dont {df['clinical_case'].isna().sum()} sans cas clinique")

In [ ]:
# Nombre de bonnes réponses par question
splits["train"]["correct_answers"].str.split(",").map(len).value_counts().sort_index().rename_axis("bonnes réponses").to_frame("questions")

# Nettoyage

In [ ]:
def clean_text(text: str) -> str:
    """Retire les espaces en début / fin de ligne et les lignes vides en trop."""
    text = "\n".join(line.strip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip()


OPTION_COLUMNS = [f"answer_{letter}" for letter in LETTERS]
for name, df in splits.items():
    splits[name] = df.assign(**{col: df[col].map(clean_text) for col in ["question"] + OPTION_COLUMNS})

In [ ]:
# Questions sans cas clinique uniquement (celles avec cas clinique sont réservées au DPO)
for name, df in splits.items():
    n_before = len(df)
    df = df[df["clinical_case"].isna()]
    splits[name] = df.drop_duplicates(subset="question")
    print(f"{name}: {n_before} rows, {len(df)} sans cas clinique, {len(splits[name])} après dédoublonnage")

In [ ]:
# Options « méta » qui renvoient aux autres options (« Aucune des propositions précédentes ») : sans la liste
# complète, elles n'ont pas de sens. Même filtre que pour le DPO.
META_OPTION = re.compile(
    r"\b(?:propositions?|réponses?)\s+(?:ci-dessus|précédentes?|[A-E])\b"
    r"|\b(?:aucune|toutes)\s+(?:les|des|de\s+ces)\s+(?:propositions|réponses)",
    re.IGNORECASE,
)

for name, df in splits.items():
    meta = df[OPTION_COLUMNS].apply(lambda col: col.str.contains(META_OPTION)).any(axis=1)
    too_short = df[OPTION_COLUMNS].apply(lambda col: col.str.len() < MIN_OPTION_CHARS).any(axis=1)
    splits[name] = df[~meta & ~too_short]
    print(f"{name}: {meta.sum()} questions avec option méta, {too_short.sum()} avec option trop courte, {len(splits[name])} rows")

# Réponse attendue

La réponse est la liste à puces des bonnes réponses, dans l'ordre des options (A à E) : **exactement le format** des réponses `chosen` du DPO français.

In [ ]:
def to_answer(row) -> str:
    """Liste à puces des bonnes réponses, dans l'ordre A à E."""
    correct = sorted(letter.strip().lower() for letter in row["correct_answers"].split(","))
    return "\n".join(f"- {row[f'answer_{letter}']}" for letter in correct)


for name, df in splits.items():
    splits[name] = df.assign(answer=df.apply(to_answer, axis=1))

example = splits["train"].iloc[0]
print(f"QUESTION : {example['question']}\n\nRÉPONSE :\n{example['answer']}")

# Échantillonnage

In [ ]:
# Découpage d'origine de MediQAl conservé
for name, size in {"train": N_TRAIN, "validation": N_VALIDATION, "test": N_TEST}.items():
    splits[name] = splits[name].sample(n=min(size, len(splits[name])), random_state=SEED)
    print(f"{name}: {len(splits[name])} rows")

# Build SFT JSONL file from dataset

In [ ]:
# Même system prompt que le SFT mcqu
SYSTEM_PROMPT = "Tu es un assistant médical expert. Réponds de façon précise et concise."


def to_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]


# Un fichier JSONL par split : ../data/sft_mediqal_mcqm_{train,validation,test}.jsonl
sft_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(source="MediQAl-mcqm", lang="fr", golden_answer=df["correct_answers"], messages=df.apply(to_messages, axis=1))
    sft_ds[name] = Dataset.from_pandas(
        df[["id", "source", "medical_subject", "question_type", "golden_answer", "lang", "messages"]], preserve_index=False
    )
    sft_ds[name].to_json(f"../data/sft_mediqal_mcqm_{name}.jsonl", force_ascii=False)

sft_ds["train"][0]["messages"]

In [ ]:
print(sft_ds)

# Anonymisation des données

On n'anonymise que les messages **user** (`roles=["user"]`). Les réponses sont des options de QCM rédigées par les enseignants, sans donnée patient : Presidio y prend des termes médicaux pour des noms (« Angiographie », « Érythrose palmaire »…) et effacerait ce que le modèle doit apprendre.

In [ ]:
# Anonymisation de chaque split : ../data/anonymized/sft_mediqal_mcqm_{split}.anonymized.jsonl
reports = {
    name: run(f"../data/sft_mediqal_mcqm_{name}.jsonl", lang="fr", output_dir="../data/anonymized", roles=["user"])
    for name in sft_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

In [ ]:
# DataFrame des messages anonymisés (hors system) et contexte de chaque balise <PERSON>, pour vérification
rows = []
for name in sft_ds:
    with open(f"../data/anonymized/sft_mediqal_mcqm_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for message in record["messages"]:
                if message["role"] != "system":
                    rows.append({"split": name, "id": record["id"], "role": message["role"], "content": message["content"]})
contents_df = pd.DataFrame(rows)


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


occurrences_df = contents_df[["split", "id", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

# Contrôle final

In [ ]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement
DATA_FILES = {name: f"../data/anonymized/sft_mediqal_mcqm_{name}.anonymized.jsonl" for name in sft_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

# Aucune question commune avec le DPO mcqm (tous splits confondus)
dpo_ids = set()
for name in ["train", "validation", "test"]:
    with open(f"../data/dpo_mediqal_mcqm_{name}.jsonl", encoding="utf-8") as f:
        dpo_ids |= {json.loads(line)["id"] for line in f}
sft_ids = {id_ for split in final_ds.values() for id_ in split["id"]}
assert not sft_ids & dpo_ids, f"{len(sft_ids & dpo_ids)} questions communes avec le DPO"
print(f"OK : aucune des {len(sft_ids)} questions SFT n'est dans le DPO ({len(dpo_ids)} questions)")

# Limites

- **Questions sans cas clinique** : connaissances générales (« Parmi les propositions suivantes, concernant la grippe… »), moins proches du triage que les cas cliniques, réservés au DPO.
- **Options absentes du prompt**, comme pour le SFT `mcqu` : le modèle doit produire les bonnes réponses sans voir les propositions.